# Raster Preprocessing & Point-level Feature ExtractionThis notebook provides two workflows for extracting covariate values at point locations from a set of aligned GeoTIFFs.## Workflows| Workflow | When to use ||----------|-------------|| **A — Two-step** | Preprocess rasters once → extract from preprocessed files repeatedly || **B — One-step** | Normalise on-the-fly during extraction (simpler, but re-normalises every run) |Both workflows produce the same output: a Parquet file where each row is a point and each covariate raster contributes one column of median/IQR-normalised values.```Workflow A                          Workflow B──────────                          ──────────Raw GeoTIFFs                        Raw GeoTIFFs     │                                   │     ▼                                   │preprocess_raster_folder()               │     │                                   │     ▼                                   ▼Normalised GeoTIFFs          extract_points_normalized()     │                              │     ▼                              ▼extract_points_preprocessed()   features.parquet     │     ▼features.parquet```

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import rowcol
from rasterio.warp import transform as rio_transform
from pathlib import Path
from typing import Optional, Tuple, List, Dict, Union

## 2. Configuration

In [ ]:
# ── Input rasters ─────────────────────────────────────────
RAW_RASTER_DIR       = "data/covariates/raw"
PROCESSED_RASTER_DIR = "data/covariates/processed"

# ── CSV splits ────────────────────────────────────────────
CV_CSV_PATH   = "data/splits/cv_split.csv"
TEST_CSV_PATH = "data/splits/test_split.csv"

# ── Output features ──────────────────────────────────────
CV_PARQUET_PATH   = "data/features/cv_point_features.parquet"
TEST_PARQUET_PATH = "data/features/test_point_features.parquet"

# ── Options ───────────────────────────────────────────────
RASTER_GLOB   = "*.tif"
LOG_TRANSFORM = False     # True only if rasters are NOT already log-transformed
COORDS_CRS    = "EPSG:4326"

# ── Required CSV columns ─────────────────────────────────
KEEP_COLS = ["Longitude", "Latitude", "fold", "cluster_id", "mod_label", "label"]

## 3. Grid & Coordinate HelpersUtilities shared by both workflows: grid-alignment checks, coordinate reprojection, and column renaming.

In [ ]:
def _grid_signature(src: rasterio.io.DatasetReader):
    """Compact grid fingerprint for equality checks."""
    return (str(src.crs), tuple(src.transform)[:6], src.width, src.height, src.nodata)


def _same_grid(sig_a, sig_b, atol: float = 1e-9) -> bool:
    """Check whether two grid signatures match (CRS, shape, transform)."""
    crs_a, t_a, w_a, h_a, _ = sig_a
    crs_b, t_b, w_b, h_b, _ = sig_b
    if crs_a != crs_b or w_a != w_b or h_a != h_b:
        return False
    return np.allclose(np.array(t_a), np.array(t_b), atol=atol, rtol=0)


def _coords_to_rc(transform, xs, ys, op=np.floor):
    """Vectorised world → row/col conversion."""
    r, c = rowcol(transform, xs, ys, op=op)
    return np.asarray(r, dtype=np.int64), np.asarray(c, dtype=np.int64)


def to_xy(df: pd.DataFrame) -> pd.DataFrame:
    """Rename Longitude/Latitude → x/y and drop rows with invalid coordinates."""
    d = df.copy().rename(columns={"Longitude": "x", "Latitude": "y"})
    valid = d["x"].notna() & d["y"].notna() & np.isfinite(d["x"]) & np.isfinite(d["y"])
    print(f"[to_xy] Keeping {valid.sum()} / {len(d)} rows with valid coordinates")
    return d[valid].copy()

## 4. Band NormalisationReads a single band, cleans extreme values, optionally log-transforms, and applies **median / IQR** normalisation.  NaN pixels are preserved (not filled).

In [ ]:
def _read_and_normalize_band(
    src: rasterio.io.DatasetReader,
    log_transform: bool = False,
    offset: float = 1e-6,
) -> Tuple[np.ndarray, Dict[str, float]]:
    """
    Read band 1, normalise to (x − median) / IQR, leave NaNs untouched.

    Returns (normalised_array, stats_dict).
    """
    m = src.read(1, masked=True).astype("float32")
    arr = np.array(m.filled(np.nan), dtype="float32")

    # drop absurd magnitudes
    arr[(arr < -1e16) | (arr > 1e16)] = np.nan

    # optional log-transform (only for non-log rasters)
    if log_transform:
        pos = arr > 0
        if np.any(pos):
            arr[pos] = np.log(arr[pos] + offset)

    v = arr[np.isfinite(arr)]
    if v.size == 0:
        return np.zeros_like(arr, dtype="float32"), {
            "median": np.nan, "iqr": np.nan, "q1": np.nan, "q3": np.nan,
        }

    med = float(np.median(v))
    q1  = float(np.percentile(v, 25))
    q3  = float(np.percentile(v, 75))
    iqr = max(q3 - q1, 1e-8)

    arr_norm = (arr - med) / iqr

    return arr_norm.astype("float32"), {"median": med, "iqr": iqr, "q1": q1, "q3": q3}

---## Workflow A — Two-step### Step 1: Preprocess RastersNormalise each raster and write the result to a new GeoTIFF. This only needs to be done once; subsequent extractions read from the preprocessed files.

In [ ]:
def preprocess_raster_folder(
    in_dir: str,
    out_dir: str,
    pattern: str = "*.tif",
    log_transform: bool = False,
    overwrite: bool = False,
) -> Dict[str, Dict[str, float]]:
    """
    Normalise all rasters in `in_dir` and write to `out_dir`.

    Returns a dict mapping raster name → normalisation statistics.
    """
    in_dir  = Path(in_dir)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    rasters = sorted(in_dir.glob(pattern))
    if not rasters:
        raise FileNotFoundError(f"No rasters matched {pattern} in {in_dir}")

    stats_all = {}

    for p in rasters:
        out_path = out_dir / p.name

        if out_path.exists() and not overwrite:
            print(f"[skip] {out_path} exists")
            continue

        with rasterio.open(p) as src:
            arr_norm, stats = _read_and_normalize_band(src, log_transform=log_transform)
            stats_all[p.stem] = stats

            profile = src.profile.copy()
            profile.update(dtype="float32", nodata=src.nodata)

            with rasterio.open(out_path, "w", **profile) as dst:
                dst.write(arr_norm, 1)

        print(f"  {p.name} → median={stats['median']:.4f}, IQR={stats['iqr']:.4f}")

    print(f"[preprocess] {len(rasters)} raster(s) processed → {out_dir}")
    return stats_all

### Step 2: Extract Points from Preprocessed RastersNo normalisation here — values are read directly from the already-preprocessed GeoTIFFs.

In [ ]:
def extract_points_preprocessed(
    raster_dir: str,
    coords: pd.DataFrame,
    lon_col: str = "x",
    lat_col: str = "y",
    pattern: str = "*.tif",
    check_same_grid: bool = True,
    coords_crs: Union[str, None] = "EPSG:4326",
    index_op=np.floor,
) -> pd.DataFrame:
    """
    Extract raster values at point locations (no normalisation).

    Assumes all rasters share the same grid/CRS and are already normalised.
    Returns the input DataFrame with one new column per raster.
    """
    rasters = sorted(Path(raster_dir).glob(pattern))
    if not rasters:
        raise FileNotFoundError(f"No rasters matched {pattern} in {raster_dir}")

    # reference grid from the first raster
    with rasterio.open(rasters[0]) as ref:
        sig0 = _grid_signature(ref)
        transform0 = ref.transform
        H0, W0 = ref.height, ref.width
        ref_crs = ref.crs

    xs = coords[lon_col].to_numpy(dtype=float)
    ys = coords[lat_col].to_numpy(dtype=float)

    # reproject if needed
    if coords_crs is not None and str(ref_crs) != str(coords_crs):
        xs_list, ys_list = rio_transform(coords_crs, ref_crs, xs.tolist(), ys.tolist())
        xs, ys = np.asarray(xs_list, float), np.asarray(ys_list, float)

    rows, cols = _coords_to_rc(transform0, xs, ys, op=index_op)
    in_bounds = (rows >= 0) & (rows < H0) & (cols >= 0) & (cols < W0)
    if not in_bounds.all():
        print(f"[extract] Warning: {(~in_bounds).sum()} coords out of bounds → NaN")

    out = coords.copy()

    for p in rasters:
        with rasterio.open(p) as src:
            if check_same_grid and not _same_grid(_grid_signature(src), sig0):
                raise RuntimeError(f"Grid mismatch: {p.name}. Align rasters first.")
            band = src.read(1).astype("float32")
            vals = np.full(rows.shape, np.nan, dtype="float32")
            vals[in_bounds] = band[rows[in_bounds], cols[in_bounds]]
            out[p.stem] = vals

    return out

---## Workflow B — One-step Extract + NormaliseReads raw rasters, normalises each band on-the-fly, and extracts point values in a single pass. Simpler but re-computes statistics on every run.

In [ ]:
def extract_points_normalized(
    raster_dir: str,
    coords: pd.DataFrame,
    lon_col: str = "x",
    lat_col: str = "y",
    pattern: str = "*.tif",
    log_transform: bool = False,
    return_stats: bool = False,
    check_same_grid: bool = True,
    coords_crs: Union[str, None] = "EPSG:4326",
    index_op=np.floor,
) -> Tuple[pd.DataFrame, Optional[Dict[str, Dict[str, float]]]]:
    """
    Extract + normalise in one step.

    Returns (features_df, stats_dict | None).
    """
    rasters = sorted(Path(raster_dir).glob(pattern))
    if not rasters:
        raise FileNotFoundError(f"No rasters matched {pattern} in {raster_dir}")

    with rasterio.open(rasters[0]) as ref:
        sig0 = _grid_signature(ref)
        transform0 = ref.transform
        H0, W0 = ref.height, ref.width
        ref_crs = ref.crs

    xs = coords[lon_col].to_numpy(dtype=float)
    ys = coords[lat_col].to_numpy(dtype=float)

    if coords_crs is not None and str(ref_crs) != str(coords_crs):
        xs_list, ys_list = rio_transform(coords_crs, ref_crs, xs.tolist(), ys.tolist())
        xs, ys = np.asarray(xs_list, float), np.asarray(ys_list, float)

    rows, cols = _coords_to_rc(transform0, xs, ys, op=index_op)
    in_bounds = (rows >= 0) & (rows < H0) & (cols >= 0) & (cols < W0)
    if not in_bounds.all():
        print(f"[extract] Warning: {(~in_bounds).sum()} coords out of bounds → NaN")

    out = coords.copy()
    stats_all = {}

    for p in rasters:
        with rasterio.open(p) as src:
            if check_same_grid and not _same_grid(_grid_signature(src), sig0):
                raise RuntimeError(f"Grid mismatch: {p.name}. Align rasters first.")
            band_norm, st = _read_and_normalize_band(src, log_transform=log_transform)
            stats_all[p.stem] = st

            vals = np.full(rows.shape, np.nan, dtype="float32")
            vals[in_bounds] = band_norm[rows[in_bounds], cols[in_bounds]]
            out[p.stem] = vals

    return out, (stats_all if return_stats else None)

## 5. Driver FunctionA convenience wrapper that reads a CSV, validates columns, converts coordinates, runs extraction via either workflow, and saves the result as Parquet.

In [ ]:
def run_extraction_and_save(
    csv_path: str,
    raster_dir: str,
    out_parquet_path: str,
    raster_glob: str = "*.tif",
    coords_crs: str = "EPSG:4326",
    workflow: str = "preprocessed",
    log_transform: bool = False,
) -> str:
    """
    End-to-end extraction driver.

    Parameters
    ----------
    workflow : "preprocessed" (Workflow A) or "normalized" (Workflow B)
    log_transform : only used when workflow="normalized"

    Returns the output parquet path.
    """
    df = pd.read_csv(csv_path)
    for c in KEEP_COLS:
        if c not in df.columns:
            raise ValueError(f"Missing column '{c}' in {csv_path}")

    base = to_xy(df[KEEP_COLS].copy())
    meta_cols = [c for c in ["x", "y", "fold", "cluster_id", "mod_label", "label"] if c in base.columns]

    if workflow == "preprocessed":
        feat_df = extract_points_preprocessed(
            raster_dir=raster_dir,
            coords=base[meta_cols].copy(),
            pattern=raster_glob,
            coords_crs=coords_crs,
        )
    elif workflow == "normalized":
        feat_df, _ = extract_points_normalized(
            raster_dir=raster_dir,
            coords=base[meta_cols].copy(),
            pattern=raster_glob,
            log_transform=log_transform,
            coords_crs=coords_crs,
        )
    else:
        raise ValueError(f"Unknown workflow: {workflow!r}. Use 'preprocessed' or 'normalized'.")

    Path(out_parquet_path).parent.mkdir(parents=True, exist_ok=True)
    feat_df.to_parquet(out_parquet_path, index=False)
    print(f"[save] {out_parquet_path}  shape={feat_df.shape}")
    return out_parquet_path

---## 6. Run: Workflow A (preprocess → extract)

In [ ]:
# Step 1: Preprocess (run once)
stats = preprocess_raster_folder(
    in_dir=RAW_RASTER_DIR,
    out_dir=PROCESSED_RASTER_DIR,
    pattern=RASTER_GLOB,
    log_transform=LOG_TRANSFORM,
    overwrite=False,
)

# Step 2: Extract CV features
run_extraction_and_save(
    csv_path=CV_CSV_PATH,
    raster_dir=PROCESSED_RASTER_DIR,
    out_parquet_path=CV_PARQUET_PATH,
    raster_glob=RASTER_GLOB,
    coords_crs=COORDS_CRS,
    workflow="preprocessed",
)

# Step 3: Extract test features
run_extraction_and_save(
    csv_path=TEST_CSV_PATH,
    raster_dir=PROCESSED_RASTER_DIR,
    out_parquet_path=TEST_PARQUET_PATH,
    raster_glob=RASTER_GLOB,
    coords_crs=COORDS_CRS,
    workflow="preprocessed",
)

## 7. Run: Workflow B (one-step extract + normalise)

In [ ]:
# Uncomment to use Workflow B instead of A
# Extracts and normalises in a single pass from raw rasters.

# run_extraction_and_save(
#     csv_path=CV_CSV_PATH,
#     raster_dir=RAW_RASTER_DIR,
#     out_parquet_path=CV_PARQUET_PATH,
#     raster_glob=RASTER_GLOB,
#     coords_crs=COORDS_CRS,
#     workflow="normalized",
#     log_transform=LOG_TRANSFORM,
# )
#
# run_extraction_and_save(
#     csv_path=TEST_CSV_PATH,
#     raster_dir=RAW_RASTER_DIR,
#     out_parquet_path=TEST_PARQUET_PATH,
#     raster_glob=RASTER_GLOB,
#     coords_crs=COORDS_CRS,
#     workflow="normalized",
#     log_transform=LOG_TRANSFORM,
# )

## 8. (Optional) Inspect Extracted Features

In [ ]:
cv_path = Path(CV_PARQUET_PATH)
if cv_path.exists():
    df = pd.read_parquet(cv_path)
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")

    meta = ["x", "y", "fold", "cluster_id", "mod_label", "label"]
    cov_cols = [c for c in df.columns if c not in meta]
    print(f"Covariates ({len(cov_cols)}): {cov_cols}")

    print(f"\nLabel distribution:")
    print(df["label"].value_counts())

    print(f"\nFold distribution:")
    print(df["fold"].value_counts().sort_index())

    print(f"\nCovariate summary (first 5):")
    print(df[cov_cols[:5]].describe().round(3).T)

    # NaN fraction per covariate
    nan_frac = df[cov_cols].isna().mean().sort_values(ascending=False)
    if nan_frac.max() > 0:
        print(f"\nCovariates with missing values:")
        print(nan_frac[nan_frac > 0])
    else:
        print(f"\nNo missing values in any covariate.")
else:
    print(f"No features found at {cv_path}. Run extraction first.")